In [17]:
import numpy as np
import pandas as pd
import gc

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder

import xgboost as xgb

In [18]:
train = pd.read_csv("playground-series-s6e4/train.csv")
test = pd.read_csv("playground-series-s6e4/test.csv")

TARGET = "Irrigation_Need"

MAP = {"Low":0, "Medium":1, "High":2}
INV_MAP = {0:"Low",1:"Medium",2:"High"}

y = train[TARGET].map(MAP)
train.drop(columns=[TARGET], inplace=True)

test_ids = test["id"]

train.head() , test.head()

(   id Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  \
 0   0     Loamy     4.92          32.58            1.01   
 1   1      Clay     7.08          56.61            0.44   
 2   2      Clay     5.69          27.71            0.81   
 3   3     Sandy     5.65          13.32            1.33   
 4   4      Clay     7.96          59.14            0.38   
 
    Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  \
 0                     3.05          15.01     50.61       725.99   
 1                     2.00          22.92     67.86       985.66   
 2                     2.83          26.97     92.22      2201.70   
 3                     0.87          13.32     61.57      1357.33   
 4                     0.96          20.22     91.11      1538.20   
 
    Sunlight_Hours  Wind_Speed_kmh  Crop_Type Crop_Growth_Stage  Season  \
 0            5.90           16.79  Sugarcane            Sowing    Zaid   
 1            6.98            3.39      Wheat        Vegetative  Kharif 

In [19]:
cat_cols = train.select_dtypes(include=["object", "string"]).columns

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]]).astype(str)
    le.fit(combined)
    
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

In [20]:
#  RAW FEATURES
X_raw = train.copy()
X_test_raw = test.copy()

#  LOG FEATURES
X_log = train.copy()
X_test_log = test.copy()

for col in X_log.columns:
    if col != "id":
        X_log[col] = np.log1p(X_log[col])
        X_test_log[col] = np.log1p(X_test_log[col])

#  BIN FEATURES
X_bin = train.copy()
X_test_bin = test.copy()

for col in X_bin.columns:
    if col != "id":
        X_bin[col] = pd.cut(X_bin[col], bins=10, labels=False)
        X_test_bin[col] = pd.cut(X_test_bin[col], bins=10, labels=False)

print("Feature sets created")

Feature sets created


In [21]:
from xgboost.callback import EarlyStopping
from xgboost import XGBClassifier
import cupy as cp

def train_xgb(X, X_test, y):
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    oof = np.zeros((len(X), 3))
    test_preds = np.zeros((len(X_test), 3))
    
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"Fold {fold+1}")
        
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        
        X_tr_gpu = cp.array(X_tr)
        X_val_gpu = cp.array(X_val)
        model = xgb.XGBClassifier(
            n_estimators=800,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.8,
            early_stopping_rounds=50,
            colsample_bytree=0.8,
            objective="multi:softprob",
            num_class=3,
            tree_method="hist",
            device="cuda",   # GPU
            eval_metric="mlogloss",
            random_state=42
        )
        
        model.fit(
            X_tr_gpu, y_tr,
            eval_set=[(X_val_gpu, y_val)],
            verbose=False
        )
        
        oof[val_idx] = model.predict_proba(X_val_gpu)
        test_preds += model.predict_proba(X_test) / 5
    
    return oof, test_preds

In [ ]:
# Equal voting
oof_final = (oof1 + oof2 + oof3) / 3
test_final = (pred1 + pred2 + pred3) / 3
oof_labels = oof_final.argmax(axis=1)

score = balanced_accuracy_score(y, oof_labels)
print("Final Balanced Accuracy:", score)
df["temp_x_moist"] = df["Temperature_C"] * df["Soil_Moisture"]
df["rain_x_irrig"] = df["Rainfall_mm"] + df["Previous_Irrigation_mm"]

In [ ]:
final_pred = (
    0.4 * xgb_pred +
    0.3 * lgb_pred +
    0.3 * cat_pred
)

submission = pd.DataFrame({
    "id": test_ids,
    "Irrigation_Need": [INV_MAP[p] for p in final_preds]
})

submission.to_csv("submission_2.csv", index=False)
print("submission_2.csv saved")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y, oof_labels)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Low","Medium","High"],
            yticklabels=["Low","Medium","High"])

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()